# **Lab:** Security in Computer Networks

**Student name:** Luciana Anchatuña & Sebastian Pinchao


### **Topics**
- Confidentiality
- Public Key Cryptography (RSA)
- Digital Signatures

In [1]:
# Install cryptography library (only first time)
# This library provides tools for encryption, decryption, signatures, etc.
%pip install cryptography

Exception ignored in: <function _afterFork at 0x1079bbba0>
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/logging/__init__.py", line 245, in _afterFork
    def _afterFork():
KeyboardInterrupt: 


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 6.1 MB/s  0:00:00 eta 0:00:01

[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import RSA module for asymmetric cryptography
from cryptography.hazmat.primitives.asymmetric import rsa

# Import serialization tools (used for saving/loading keys, not used deeply here)
from cryptography.hazmat.primitives import serialization

# Import padding schemes (needed for secure encryption)
from cryptography.hazmat.primitives.asymmetric import padding

# Import hashing algorithms (SHA256, etc.)
from cryptography.hazmat.primitives import hashes

## **🔐 Part 1:** RSA (Public Key Cryptography)

- Public key → shared
- Private key → secret
- Encrypt with public key, decrypt with private key


In [4]:
# Generate PRIVATE key
# public_exponent=65537 is a standard safe value used in RSA
# key_size=2048 means 2048-bit key (secure for most purposes)
private_key = rsa.generate_private_key(
    public_exponent=65537,   # mathematical parameter used in RSA
    key_size=2048            # size of the key in bits (larger = more secure but slower)
)

# Generate PUBLIC key from private key
# In RSA, both keys are mathematically linked
public_key = private_key.public_key()

print("Keys generated successfully")

Keys generated successfully


In [5]:
# Export public key to PEM
pem_pub = public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo
).decode()

pem_pub

'-----BEGIN PUBLIC KEY-----\nMIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAlopG7gNcUVYnmZHE554F\n4RwZgdDzTvYjX68ntAAmuiOWABF8FkhUm+ks1oVNChgipk7AQwwMnLvLKASmIlGb\nR9rTeIAWcloxYR1FHKJwp4QZY8HMdnSUKrH7U+nZ/WLbD0Ga/WA/BxZdbdKUya2Q\nlRRA2KcMFuVi8VXQ6Y6hbHOaz0bxjYIBTiF0MzkJUrgmVikAkTBGvsBwlLhYA9cg\nyPKH5gR9R/m1140vxNcnLXUcEQVhQipwoo6oj94U1vVFzt91LYoYFbpN3mG4Ym91\nli+3Diz7jkAogIIkal3Dv+AW76waHP/cLOCx3tIa/xSo0ut4HPn/7dSQTSCgsD80\newIDAQAB\n-----END PUBLIC KEY-----\n'

In [16]:
receiver_pem = (
    '-----BEGIN PUBLIC KEY-----\n'
    'MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAlnuoapUO/+0lTw8iQRP9\n'
    'iztxrde0wvUqz5nn4oE/uB7WpBc46ZWtMMYFgaG2Dqci4W8BvUQ42VwqvYwrDRW7\n'
    'UbPVMsfI0ENZ5NsPFeSsHywZ64fssx8hYSxCRD0u+fpU7zaYJDog+Q4pPeTXqv57\n'
    'W3roY8Ia+4QMDyuq6bEkX2aP5PgC5Q1G5VLV/gUM9+FNJG3cNOwJTwUbVIu68iC2\n'
    'myIuAqJ25dSG7vqdawkzBneSEU4N/bd4M/gz1PVE8pxTvvdeOn3H62vxUjexyekF\n'
    'Nt0alWaHxy3mmkU0qUR8D4MSt3YGUXFi8Se/T0wT/T1vORhDiOkdh3qVMVDSnjkg\n'
    'vwIDAQAB\n'
    '-----END PUBLIC KEY-----\n'
)

receiver_public_key = serialization.load_pem_public_key(
    receiver_pem.encode()
)

print("Public key loaded successfully!")

Public key loaded successfully!


In [17]:
import base64

# Define message to encrypt
# .encode() converts string → bytes (required for crypto functions)
message = "Hello secure world".encode()

# Encrypt using PUBLIC key
encrypted = receiver_public_key.encrypt(
    message,  # data to encrypt (must be bytes)
    padding.OAEP(  # OAEP = Optimal Asymmetric Encryption Padding (secure padding)
        mgf=padding.MGF1(algorithm=hashes.SHA256()),  # MGF1 = Mask Generation Function
        algorithm=hashes.SHA256(),  # hash algorithm used internally
        label=None  # optional label (not needed here)
    )
)

print("Encrypted (Python bytes object):", encrypted) # This will show as a bytes object (not human-readable). ASCIIs are not suitable for binary data, so we will convert it to hex and base64 for display.
print("Encrypted (hex):", encrypted.hex())
print("Encrypted (base64):", base64.b64encode(encrypted).decode()) # Standard way to represent binary data as text

Encrypted (Python bytes object): b'\x1d>\x90a{\xe4\x99\x89\x83\xbc~\xb8\xf7C!\xfajV\x95#\xec#\xd6zP\xed\xcc\x03;F\x00J\x9eE\xb5\xae\xfc\x06Ub"\xb9\xa8|\xdd\n\xe9\xf0\xae\x9a+9\xbd;\xa9}hx\xc7\xf5\xd6\x16<\xe9X\xb7oT\xf8Q\x10\x0fP\x87\xd8\xdf\xd8\x9cj\xcdr\xa7N\x83\xaa\xc3N\xc3L\x07\xabml\xf8\x0eS{\xa1n 5)\xfd*L\x12\xad\xfb\x87NK\x06I\x96\xa7\xff*(\xcf\x96\xbf\x16\x99\x81\x84\x86\x15\xbf\xf7.+N\x87\x1a\xb7\x96\x01\xb4\x7f2\xf6"iA\x0c\x97x\xa9\xbe\xdb\xff~2\xc96#\x1b\x9e\x88I\xe8+\xce\xcf\x98\x92-\xb8\x04\xde\xcdx,\xa3>\xe3\xcf\xee\xd3\xac\x9d9\x9e2\xeb\xad\x04\xea\xe7\x0b\x0f?\xf9\xb7\xac\xbfpP\x1fE\xdbZ\xfev\x19P\xea-\x90U\xd8\xf6\xd4\x83O\x04x\x01\xbf\xf5V\xbb\\*9\xdc\x065\x0c\xdb\xd44\xbd\x01\x92\xc4\xa81q\xddT5GG_\xf7\xc3s\x0f\xc1$\xbf6\xe1\xbe7'
Encrypted (hex): 1d3e90617be4998983bc7eb8f74321fa6a569523ec23d67a50edcc033b46004a9e45b5aefc06556222b9a87cdd0ae9f0ae9a2b39bd3ba97d6878c7f5d6163ce958b76f54f851100f5087d8dfd89c6acd72a74e83aac34ec34c07ab6d6cf80e537ba16e203529fd2a4c12adfb874e4b0

In [22]:
# Decrypt using PRIVATE key
encrypted = base64.b64decode(
    "SyZ3lnwljM8ejrT66vikm71CFyUFcaFR7ICdM7is2aO7irupBZuqMDICZV8I6PLloKyW2KOR7I6ICyOQK3uCnbwSSQgNnsYVnchD5P/HJp1CaNan1RtXy4vgzC6evHiADOPgLcuet9g+UZFS9fK/XLT/rT5NenUbL/mn+DkNVmCMa9B5m+Acd9EOl9ei8W47zRtjK/df35jLKONpIi1EFwq1WtFBXB60230QXTDu0ZXdnT5ynbpWWvALure6tU/Jrf0JdS6lQSDGMqZOXwjuf4+MSMPOfT6ysVmFu5rAsCipggILB5iN2eXavrKKxksEXWVFQDVyH/G04JWLyWTILA=="
)
decrypted = private_key.decrypt(
    encrypted,  # encrypted data
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

# Convert bytes → string using .decode()
print("Decrypted:", decrypted.decode())

Decrypted: ola q hace


When you:

1. Take a message  
2. Encrypt it with the **receiver’s public key**  
3. Encode it in **Base64**

You already achieved **confidentiality**:  
Only the holder of the **private key** can decrypt it.

> ⚠️ **Base64 is NOT security**, it is just an encoding format to represent binary data as text.

The real problem (without certificates)

The key question is:

> ❗ How do you know that the **public key** actually belongs to the right person?

Without certificates, this can happen:

- An attacker gives you **their public key**
- You encrypt the message using that key
- The attacker decrypts it with their private key
- Then re-encrypts it using the real recipient’s key

*Result:* your communication was intercepted without you noticing.

This is known as a **Man-in-the-Middle** attack.

So… what are certificates for?

A certificate (e.g., `X.509`) is used to:

**Bind:**
- A **public key**
- To a **real identity** (person, server, organization)

**It also includes:**
- A trusted signature (from a Certificate Authority)
- Owner information
- Expiration/validity dates

Who guarantees this?

A **Certificate Authority (CA)** such as:

- Let's Encrypt  
- DigiCert  

These entities sign certificates and essentially say:

> “Yes, this public key belongs to this entity.”

## **✍️ Part 2:** Digital Signatures

- Sign with PRIVATE key
- Verify with PUBLIC key


In [21]:
# Create digital signature
signature = private_key.sign(
    message,  # original message
    padding.PSS(  # Probabilistic Signature Scheme (secure signature padding)
        mgf=padding.MGF1(hashes.SHA256()),  # mask generation function
        salt_length=padding.PSS.MAX_LENGTH  # random salt for security
    ),
    hashes.SHA256()  # hash algorithm used before signing
)

print("Signature created")
print("Signature (base64):", base64.b64encode(signature).decode())

Signature created
Signature (base64): Lq8I2LwfIivB/4Wox6UBRyyiIINcV/FsiO0S2J+WqxqRZD3dGTN7aafTKN3fTyvHx1Uf13dTdvh2VfV08GNfIgUxXDkvOxOBZdAG+z1O9gghbv1Sb8CYAARaiQL+OT2svYnff7N1lHGdTFJnPelVAzU6qS152UqMoQLhVFXfx5fEdzQWP6aN2HuX+N8advc0R51o5O41RPpDsDbNxeijQctoFbBlL/c5H4J7Co1VXXAzuMeTx51fex8b5yHPAAOgWZ2Amm2VS1xOnTtIKTv6d5k2ziUqzAjAPIi85gW6APz3IN4Ikh0jeKefZsDAL6iH/whKLVlr4GzGzl8H7FlKxg==


In [24]:
received_signature_b64 = """Z2/ko3Qw5yjVZtCZqjOLv/+NRRGNJ6ohnfA7qQoS+HfnRS/OimAmuSLOzzX5eohou7TunIYIA6qLHtzxwHboz2yrfrwnoSwwwCCBgOUCnzTtNxjCnK7aXXO8V4ZA7LFJTZzPsFL2TFcxt1Zz27OGfFqdphmi3qHT8oVLMl49RbH1VjYEE0BoFnEOWum2MjJyEiO+OkTc67N7u9tIVF82s8HCGSDoHg5/QOxmyVXJPDN1B/5p8xNqyx9+5BqyY9VAhrhfCOhd0fSDEA+swRJktaOs4GFyFMLEtpYnnAeHq82FRtRxySzoltoN30SR6pWBzAzpyVk8Je6WBsuqSmDCsg==
"""

signature = base64.b64decode(
    "".join(received_signature_b64.split()),
    validate=True
)
# Verify signature
try:
    receiver_public_key.verify(
        signature,
        decrypted,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    print("Signature VALID")
except:
    print("Signature INVALID")

Signature VALID


How it fits together?

In protocols like **Transport Layer Security (TLS)**:

1. The server sends its certificate  
2. The client verifies:
   - The CA signature  
   - The domain name  
3. **Only then** the public key is trusted  
4. The key is used for encryption

## **🌐 Part 3:** TLS Handshake Simulation

Real TLS:
1. Client gets server public key
2. Client creates session key
3. Client encrypts session key with public key
4. Server decrypts
5. Both use symmetric encryption


In [11]:
# Import symmetric encryption tool
from cryptography.fernet import Fernet

In [12]:
# Client generates symmetric session key
# This key will be used for fast encryption (like AES)
session_key = Fernet.generate_key()

# Create cipher object for encryption/decryption
cipher_client = Fernet(session_key)

print("Session key created")

Session key created


In [13]:
# Client encrypts session key using server PUBLIC key
encrypted_session_key = public_key.encrypt(
    session_key,
    padding.OAEP(
        mgf=padding.MGF1(hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

In [14]:
# Server decrypts session key using PRIVATE key
decrypted_session_key = private_key.decrypt(
    encrypted_session_key,
    padding.OAEP(
        mgf=padding.MGF1(hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

# Server creates cipher with shared key
cipher_server = Fernet(decrypted_session_key)


In [ ]:
# Secure communication

secure_message = cipher_client.encrypt(b"Hello TLS secure world")

print("Encrypted message:", secure_message)

print("Decrypted message:", cipher_server.decrypt(secure_message).decode())

Encrypted message: b'gAAAAABq-0lpMNV-o7sExzh5TwOQQNsTo54VHzce7wK-g8N8AQhSi3SUOWffPNC8YqxtQ0JYGo6nAr4sdRlw-iwQHFtVb7f5TTPAreS3Htkj0TXpExxN3nU='
Decrypted message: Hello TLS secure world


## **Assignment:** Secure Communication Challenge

### **Objective**
Apply all concepts from the lab:
- Confidentiality (encryption)
- Authentication (digital signatures)
- Integrity (verification)

### **Group Dynamics (Balanced Work)**

Each student must complete ALL roles:

1. **Sender**
2. **Receiver**

You will interact with at least **one other student**.

### **Task 1:** Secure Message Sending

Each student must:

1. Generate an RSA key pair
2. Share ONLY the **public key**
3. Create a message
4. Encrypt the message using the receiver’s public key
5. Sign the message using your private key

👉 Send to another student:
- Encrypted message
- Signature
- Your public key

### **Task 2:** Secure Message Receiving

Each student must:

1. Receive:
   - Encrypted message
   - Signature
   - Sender's public key

2. Decrypt the message using your private key  
3. Verify the signature using the sender’s public key  

✔ If valid → accept message  
❌ If invalid → reject message  

### **Time Limit**
- 20–25 minutes total

### **Evaluation (Checklist)**

| **Criteria** | **Score (10pts)** |
|--------|------|
| RSA encryption implemented | 1 pts |
| Digital signature implemented | 1 pts |
| Correctly decrypted & verified message | 4 pts |
| Explanation reflection questions | 4 pts |

### **Reflection Questions**

1. Why is encryption alone NOT enough for security?  
Encryption protects confidentiality, but it does not guarantee integrity or authentication. An attacker could modify an encrypted message or impersonate the sender. 
2. What is the role of digital signatures?  
Digital signatures provide authentication and integrity. The sender signs a message using their private key, and the receiver verifies the signature using the sender's public key. This helps confirm that the message has not been modified and was signed by the holder of the corresponding private key.
3. Why does TLS use a hybrid approach? 
TLS combines asymmetric and symmetric cryptography to achieve security and efficiency. Asymmetric cryptography is used for authentication and secure key establishment, while symmetric encryption provides faster data transmission.
